In [0]:
%skip
%sql
-- preview csv file
SELECT * FROM read_files('/Volumes/workspace/bronze/csvfiles/patient_clinical notes1.csv',format=>'csv') LIMIT 5;

--SELECT * FROM read_files('/Volumes/workspace/bronze/csvfiles/patient_clinical notes1.csv',format=>'csv',sep=>'|',header=>'true') LIMIT 5;

-- create streaming tables 
CREATE OR REPLACE STREAMING TABLE bronze.strm_patient_clinical_notes 
SCHEDULE EVERY 1 WEEK      -- scheduling refresh is optional
AS
SELECT * FROM STREAM read_files('/Volumes/workspace/bronze/csvfiles',format=>'csv');     -- path to folder not the file name

In [0]:
%python
# First Commit
from pyspark.sql import functions as F
checkpoint_path='/Volumes/workspace/bronze/chkpointfiles/'
df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("cloudFiles.inferColumnTypes", "true")
      .option("cloudFiles.schemaLocation", checkpoint_path)
      .load("/Volumes/workspace/bronze/csvfiles/"))

'''parsed = (df.withColumn("ingestion_ts", F.current_timestamp())
             .selectExpr(
               "_metadata.file_path as src_path",
               "note_id", "patient_id", "encounter_id", "note_text", "note_ts",
               "ingestion_ts"
             ))'''

(df.writeStream
  .format("delta")
  .option("checkpointLocation", "/Volumes/workspace/bronze/chkpointfiles/")
  .outputMode("append")
  .trigger(availableNow=True)
  .toTable("bronze.b_clinicalnotes"))  # UC table path


In [0]:
%skip
%sql
CREATE OR REPLACE STREAMING LIVE TABLE bronze.strm_patient_clinical_notes 
--SCHEDULE EVERY 1 WEEK      -- scheduling refresh is optional

Not working when run as a cell but Working when you run this as part of an ETL pipeline

In [0]:
-- preview csv file
-- SELECT * FROM read_files('/Volumes/workspace/bronze/csvfiles/patient_clinical notes1.csv',format=>'csv') LIMIT 5;  --pipeline will not

--SELECT * FROM read_files('/Volumes/workspace/bronze/csvfiles/patient_clinical notes1.csv',format=>'csv',sep=>'|',header=>'true') LIMIT 5;

-- create streaming tables 
CREATE OR REPLACE STREAMING LIVE TABLE bronze.strm_patient_clinical_notes 
SCHEDULE EVERY 1 WEEK      -- scheduling refresh is optional
AS
SELECT * FROM STREAM read_files('/Volumes/workspace/bronze/csvfiles',format=>'csv');     -- path to folder not the file name